# Sesión 02 – Fundamentos de Spark aplicados al Proyecto de Ventas

**Objetivo:** Aplicar los fundamentos de PySpark sobre el dataset real de ventas (facturas electrónicas 2021-2026), continuando desde la capa batch de la arquitectura Lambda construida en el Notebook 01.

**Dataset:** `Ventas_2021-2026.xlsx` — el mismo que cargamos, limpiamos y cacheamos en el Notebook 01.

**Temas que cubrimos en esta sesión:**
- SparkSession y configuración
- Carga de datos (Excel → pandas → Spark DataFrame)
- Transformaciones vs. Acciones (Lazy Evaluation)
- Plan de ejecución con `explain()`
- De DataFrame a RDD para procesamiento tipo MapReduce
- Aggregaciones batch sobre ventas reales

```
Arquitectura Lambda – Capa Batch
┌──────────────────────────────────────────┐
│  Ventas_2021-2026.xlsx                   │
│          │                               │
│          ▼                               │
│  [Ingesta - pandas]  ← ya hecho en NB01 │
│          │                               │
│          ▼                               │
│  [Limpieza - PySpark] ← ya hecho en NB01│
│          │                               │
│          ▼                               │
│  [Fundamentos Spark] ← ESTE NOTEBOOK    │
│          │                               │
│          ▼                               │
│  [Aggregaciones batch → Métricas]        │
└──────────────────────────────────────────┘
```

## 1. Crear la SparkSession

Replicamos la misma configuración del Notebook 01. En un proyecto real, la SparkSession viviría en un proceso compartido; aquí la recreamos por completitud académica.

> **Lazy Evaluation:** Spark no ejecuta ninguna transformación al crearla. Solo cuando llamemos a una **acción** (`.show()`, `.count()`, `.collect()`) arrancará el procesamiento real.

In [ ]:
#!pip install openpyxl

In [1]:
import pandas as pd
import warnings
import re
from operator import add
import unicodedata
warnings.filterwarnings('ignore')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType, DateType
)

# Inicializar SparkSession — misma config que Notebook 01
spark = (
    SparkSession.builder
    .appName("ProyectoVentas_Fundamentos")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession lista")
print(f"   Versión  : {spark.version}")
print(f"   App Name : {spark.sparkContext.appName}")
print(f"   Master   : {spark.sparkContext.master}")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 12:51:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/14 12:51:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ SparkSession lista
   Versión  : 4.1.1
   App Name : ProyectoVentas_Fundamentos
   Master   : local[*]


## 2. Cargar el dataset (Excel → pandas → Spark DataFrame)

Repetimos el pipeline de ingesta del Notebook 01 para que este notebook sea **autocontenido**.

| Etapa | Herramienta | Razón |
|---|---|---|
| Leer `.xlsx` | pandas | Spark no soporta Excel nativamente |
| Normalizar columnas | Python puro | Limpieza de nombres antes de Spark |
| Crear Spark DataFrame | PySpark | Para procesamiento distribuido batch |

In [2]:
EXCEL_PATH = "../data/Ventas_2021-2026.xlsx"

print("⏳ Cargando Excel con pandas...")
df_pd = pd.read_excel(EXCEL_PATH, dtype=str)
print(f"✅ Excel cargado → {df_pd.shape[0]:,} filas × {df_pd.shape[1]} columnas")

# Normalizar nombres de columnas
def normalizar_columna(nombre):
    nombre = str(nombre).strip().lower()
    nombre = unicodedata.normalize('NFD', nombre)
    nombre = ''.join(c for c in nombre if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace(' ', '_').replace('/', '_').replace('-', '_')
    return nombre

df_pd.columns = [normalizar_columna(c) for c in df_pd.columns]
print(f"✅ Columnas normalizadas: {df_pd.columns.tolist()}")

⏳ Cargando Excel con pandas...
✅ Excel cargado → 525,434 filas × 24 columnas
✅ Columnas normalizadas: ['fecha_de_emision', 'tipo', 'serie', 'numero', 'doc_entidad_tipo', 'doc_entidad_numero', 'denominacion_entidad', 'moneda', 'tipo_de_cambio', 'unidad_de_medida', 'codigo', 'descripcion', 'cantidad', 'valor_unitario', 'precio_unitario', 'descuento', 'subtotal', 'tipo_de_igv', 'igv', 'tipo_de_isc', 'isc', 'impuesto_bolsas', 'total', 'anulado']


In [3]:
# Convertir a Spark DataFrame
df = spark.createDataFrame(df_pd)

print(f"✅ Spark DataFrame creado")
print(f"   Particiones iniciales : {df.rdd.getNumPartitions()}")
print(f"   Total registros       : {df.count():,}")
print()
print("📋 Schema (todo como string — se tipará en el paso de limpieza):")
df.printSchema()

✅ Spark DataFrame creado
   Particiones iniciales : 28


26/04/14 12:56:20 WARN TaskSetManager: Stage 0 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 0:>                                                        (0 + 28) / 28]

   Total registros       : 525,434

📋 Schema (todo como string — se tipará en el paso de limpieza):
root
 |-- fecha_de_emision: string (nullable = true)
 |-- tipo: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- doc_entidad_tipo: string (nullable = true)
 |-- doc_entidad_numero: string (nullable = true)
 |-- denominacion_entidad: string (nullable = true)
 |-- moneda: string (nullable = true)
 |-- tipo_de_cambio: string (nullable = true)
 |-- unidad_de_medida: string (nullable = true)
 |-- codigo: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- cantidad: string (nullable = true)
 |-- valor_unitario: string (nullable = true)
 |-- precio_unitario: string (nullable = true)
 |-- descuento: string (nullable = true)
 |-- subtotal: string (nullable = true)
 |-- tipo_de_igv: string (nullable = true)
 |-- igv: string (nullable = true)
 |-- tipo_de_isc: double (nullable = true)
 |-- isc: string (nullable = true)
 |-

## 3. Explorar estructura y primeros registros

Revisamos el contenido real del dataset: columnas, tipos y una vista previa.

> **Nota académica:** `.show()` es una **acción** — aquí Spark ejecuta por primera vez el plan construido hasta este punto.

In [50]:
# ============================================
# CONFIGURACIÓN MANUAL DE COLUMNAS (RECOMENDADO)
# ============================================

# 1. Columna de productos (forzada manualmente)
DESC_COL = 'descripcion'  # ← Esta es la que tiene PEPSI, GATORADE, etc.

print(f"✅ Columna de productos: '{DESC_COL}'")
print(f"   Ejemplo: {df.select(DESC_COL).first()[0][:50]}...")
print()

# 2. Detectar otras columnas automáticamente
col_fecha   = [c for c in df.columns if 'fecha' in c.lower()]
col_total   = [c for c in df.columns if 'total' in c.lower()]
col_anulado = [c for c in df.columns if 'anulado' in c.lower()]
col_cliente = [c for c in df.columns if any(keyword in c.lower() for keyword in 
                ['cliente', 'razon', 'denominacion_entidad', 'proveedor', 'vendedor'])]

print("🔍 Otras columnas detectadas:")
print(f"  Fecha      : {col_fecha[0] if col_fecha else 'No encontrada'}")
print(f"  Total      : {col_total[0] if col_total else 'No encontrada'}")
print(f"  Anulado    : {col_anulado[0] if col_anulado else 'No encontrada'}")
print(f"  Cliente    : {col_cliente[0] if col_cliente else 'No encontrada'}")

# Guardar referencias
FECHA_COL   = col_fecha[0]   if col_fecha   else None
TOTAL_COL   = col_total[0]   if col_total   else None
ANULADO_COL = col_anulado[0] if col_anulado else None

print(f"\n✅ Configuración final:")
print(f"   DESC_COL   = '{DESC_COL}'")
print(f"   FECHA_COL  = '{FECHA_COL}'")
print(f"   TOTAL_COL  = '{TOTAL_COL}'")
print(f"   ANULADO_COL= '{ANULADO_COL}'")

✅ Columna de productos: 'descripcion'
   Ejemplo: GATORADE TROPICAL 500 ML - BOTELLA ...

🔍 Otras columnas detectadas:
  Fecha      : fecha_de_emision
  Total      : subtotal
  Anulado    : anulado
  Cliente    : denominacion_entidad

✅ Configuración final:
   DESC_COL   = 'descripcion'
   FECHA_COL  = 'fecha_de_emision'
   TOTAL_COL  = 'subtotal'
   ANULADO_COL= 'anulado'


26/04/14 13:24:03 WARN TaskSetManager: Stage 77 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [51]:
# Vista previa de las primeras filas — acción .show()
print("📄 Primeras 5 filas del dataset:")
df.show(5, truncate=True)

📄 Primeras 5 filas del dataset:
+----------------+----+-----+------+----------------+------------------+--------------------+------+--------------+----------------+-------------------+--------------------+--------+--------------+---------------+---------+-------------+-----------+-------------+-----------+---+---------------+--------+-------+
|fecha_de_emision|tipo|serie|numero|doc_entidad_tipo|doc_entidad_numero|denominacion_entidad|moneda|tipo_de_cambio|unidad_de_medida|             codigo|         descripcion|cantidad|valor_unitario|precio_unitario|descuento|     subtotal|tipo_de_igv|          igv|tipo_de_isc|isc|impuesto_bolsas|   total|anulado|
+----------------+----+-----+------+----------------+------------------+--------------------+------+--------------+----------------+-------------------+--------------------+--------+--------------+---------------+---------+-------------+-----------+-------------+-----------+---+---------------+--------+-------+
|      20/01/2021|  03| BQQ1|

26/04/14 13:24:21 WARN TaskSetManager: Stage 78 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## 4. Exploración con DataFrames — Transformaciones y Acciones

Aplicamos **transformaciones** (`.select()`, `.filter()`, `.withColumn()`) que construyen el plan lógico, y **acciones** (`.show()`, `.count()`) que lo ejecutan.

| Operación | Tipo | ¿Spark ejecuta? |
|---|---|---|
| `.select()` | Transformación | ❌ No |
| `.filter()` | Transformación | ❌ No |
| `.withColumn()` | Transformación | ❌ No |
| `.show()` | Acción | ✅ Sí |
| `.count()` | Acción | ✅ Sí |
| `.collect()` | Acción | ✅ Sí |

In [52]:
# TRANSFORMACIÓN: seleccionar columnas clave
cols_clave = [c for c in [FECHA_COL, TOTAL_COL, DESC_COL, ANULADO_COL] if c]
df_seleccionado = df.select(*cols_clave)  # ← solo construye el plan

print("📋 Plan construido (sin ejecutar aún):")
print(df_seleccionado)  # muestra el tipo, no los datos
print()

# ACCIÓN: ahora Spark ejecuta
print("▶️  Ejecutando con .show() (acción):")
df_seleccionado.show(5, truncate=True)

📋 Plan construido (sin ejecutar aún):
DataFrame[fecha_de_emision: string, subtotal: string, descripcion: string, anulado: string]

▶️  Ejecutando con .show() (acción):
+----------------+-------------+--------------------+-------+
|fecha_de_emision|     subtotal|         descripcion|anulado|
+----------------+-------------+--------------------+-------+
|      20/01/2021|            0|GATORADE TROPICAL...|     SI|
|      24/01/2021|            0|CONCORDIA PIÑA 30...|     SI|
|      25/01/2021|11.0169915254|PEPSI 355 ML SABO...|     NO|
|      25/01/2021|77.9661016949|CORONA EXTRA BOTE...|     NO|
|      25/01/2021|22.0338983051|PEPSI 3000 ML - B...|     NO|
+----------------+-------------+--------------------+-------+
only showing top 5 rows


26/04/14 13:24:23 WARN TaskSetManager: Stage 79 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


In [53]:
# TRANSFORMACIÓN encadenada: filtrar y seleccionar
if ANULADO_COL:
    df_validos = (
        df
        .filter(
            (F.col(ANULADO_COL).isNull()) |
            (~F.upper(F.col(ANULADO_COL)).isin("SI", "S", "1", "TRUE", "ANULADO"))
        )
        .select(*cols_clave)
    )
else:
    df_validos = df.select(*cols_clave)

# Hasta aquí: solo un plan lógico
print("📐 Transformaciones aplicadas (sin ejecutar):")
print(f"   Tipo del objeto: {type(df_validos)}")

# ACCIÓN: contar registros válidos
n_validos = df_validos.count()  # ← ACCIÓN real
print(f"\n✅ Registros válidos (sin anulados): {n_validos:,}")

📐 Transformaciones aplicadas (sin ejecutar):
   Tipo del objeto: <class 'pyspark.sql.classic.dataframe.DataFrame'>


26/04/14 13:24:27 WARN TaskSetManager: Stage 80 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.



✅ Registros válidos (sin anulados): 522,688


## 5. Limpieza y tipado — Transformaciones clave

Convertimos tipos de datos. Cada `.withColumn()` es una transformación que Spark encadena en el DAG (Grafo Acíclico Dirigido) de operaciones.

In [54]:
df_clean = df

# 1. Convertir fecha (transformación)
if FECHA_COL:
    df_clean = df_clean.withColumn(
        FECHA_COL,
        F.to_date(F.col(FECHA_COL), "dd/MM/yyyy")
    )
    print(f"✅ '{FECHA_COL}' → DateType")

# 2. Convertir total a numérico (transformación)
if TOTAL_COL:
    df_clean = df_clean.withColumn(
        TOTAL_COL,
        F.regexp_replace(F.col(TOTAL_COL), ",", "").cast("double")
    )
    print(f"✅ '{TOTAL_COL}' → DoubleType")

# 3. Filtrar anulados (transformación)
if ANULADO_COL:
    df_clean = df_clean.filter(
        (F.col(ANULADO_COL).isNull()) |
        (~F.upper(F.col(ANULADO_COL)).isin("SI", "S", "1", "TRUE", "ANULADO"))
    )
    print(f"✅ Filtro anulados aplicado")

# 4. Columnas temporales (transformaciones)
if FECHA_COL:
    df_clean = (
        df_clean
        .withColumn("anio",      F.year(F.col(FECHA_COL)))
        .withColumn("mes",       F.month(F.col(FECHA_COL)))
        .withColumn("anio_mes",  F.date_format(F.col(FECHA_COL), "yyyy-MM"))
        .withColumn("trimestre", F.quarter(F.col(FECHA_COL)))
    )
    print(f"✅ Columnas temporales: anio, mes, anio_mes, trimestre")

print()
print("⚠️  Todas las operaciones anteriores son TRANSFORMACIONES (lazy)")
print("   Spark aún no ha ejecutado nada en el cluster")

✅ 'fecha_de_emision' → DateType
✅ 'subtotal' → DoubleType
✅ Filtro anulados aplicado
✅ Columnas temporales: anio, mes, anio_mes, trimestre

⚠️  Todas las operaciones anteriores son TRANSFORMACIONES (lazy)
   Spark aún no ha ejecutado nada en el cluster


In [55]:
# ACCIÓN: verificar el resultado de la limpieza
print("▶️  Ejecutando acción .show() — aquí Spark corre el plan completo:")
df_clean.select(
    FECHA_COL, TOTAL_COL, "anio", "mes", "anio_mes", "trimestre"
).show(5)

▶️  Ejecutando acción .show() — aquí Spark corre el plan completo:
+----------------+-------------+----+---+--------+---------+
|fecha_de_emision|     subtotal|anio|mes|anio_mes|trimestre|
+----------------+-------------+----+---+--------+---------+
|      2021-01-25|11.0169915254|2021|  1| 2021-01|        1|
|      2021-01-25|77.9661016949|2021|  1| 2021-01|        1|
|      2021-01-25|22.0338983051|2021|  1| 2021-01|        1|
|      2021-01-25|11.0169915254|2021|  1| 2021-01|        1|
|      2021-01-25|      0.86667|2021|  1| 2021-01|        1|
+----------------+-------------+----+---+--------+---------+
only showing top 5 rows


26/04/14 13:24:33 WARN TaskSetManager: Stage 83 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
  File "/usr/local/lib/python3.10/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
BrokenPipeError: [Errno 32] Broken pipe


## 6. Analizar el Plan de Ejecución con `explain()`

`explain()` muestra cómo Catalyst Optimizer transforma nuestras operaciones en un plan físico eficiente.

| Plan | Descripción |
|---|---|
| **Parsed Logical Plan** | El árbol de operaciones tal como lo escribimos |
| **Analyzed Logical Plan** | Con tipos resueltos |
| **Optimized Logical Plan** | Luego de que Catalyst aplica reglas de optimización |
| **Physical Plan** | Lo que realmente ejecuta el cluster |

In [56]:
# Plan de ejecución completo de la limpieza + columnas temporales
print("🔍 Plan de ejecución del DataFrame limpio:")
print("=" * 60)
df_clean.select(
    FECHA_COL, TOTAL_COL, "anio", "mes", "anio_mes"
).explain(True)

🔍 Plan de ejecución del DataFrame limpio:
== Parsed Logical Plan ==
'Project ['fecha_de_emision, 'subtotal, 'anio, 'mes, 'anio_mes]
+- Project [fecha_de_emision#780, tipo#1, serie#2, numero#3, doc_entidad_tipo#4, doc_entidad_numero#5, denominacion_entidad#6, moneda#7, tipo_de_cambio#8, unidad_de_medida#9, codigo#10, descripcion#11, cantidad#12, valor_unitario#13, precio_unitario#14, descuento#15, subtotal#781, tipo_de_igv#17, igv#18, tipo_de_isc#19, isc#20, impuesto_bolsas#21, total#22, anulado#23, anio#782, ... 3 more fields]
   +- Project [fecha_de_emision#780, tipo#1, serie#2, numero#3, doc_entidad_tipo#4, doc_entidad_numero#5, denominacion_entidad#6, moneda#7, tipo_de_cambio#8, unidad_de_medida#9, codigo#10, descripcion#11, cantidad#12, valor_unitario#13, precio_unitario#14, descuento#15, subtotal#781, tipo_de_igv#17, igv#18, tipo_de_isc#19, isc#20, impuesto_bolsas#21, total#22, anulado#23, anio#782, ... 2 more fields]
      +- Project [fecha_de_emision#780, tipo#1, serie#2, numero

In [57]:
# Plan de una aggregación (más complejo — incluye Exchange/shuffle)
if TOTAL_COL:
    df_por_anio = df_clean.groupBy("anio").agg(
        F.sum(TOTAL_COL).alias("total_ventas"),
        F.count("*").alias("num_facturas")
    ).orderBy("anio")

    print("🔍 Plan de la aggregación por año (observa el Exchange/shuffle):")
    print("=" * 60)
    df_por_anio.explain(True)

🔍 Plan de la aggregación por año (observa el Exchange/shuffle):
== Parsed Logical Plan ==
'Sort ['anio ASC NULLS FIRST], true
+- Aggregate [anio#782], [anio#782, sum(subtotal#781) AS total_ventas#805, count(1) AS num_facturas#806L]
   +- Project [fecha_de_emision#780, tipo#1, serie#2, numero#3, doc_entidad_tipo#4, doc_entidad_numero#5, denominacion_entidad#6, moneda#7, tipo_de_cambio#8, unidad_de_medida#9, codigo#10, descripcion#11, cantidad#12, valor_unitario#13, precio_unitario#14, descuento#15, subtotal#781, tipo_de_igv#17, igv#18, tipo_de_isc#19, isc#20, impuesto_bolsas#21, total#22, anulado#23, anio#782, ... 3 more fields]
      +- Project [fecha_de_emision#780, tipo#1, serie#2, numero#3, doc_entidad_tipo#4, doc_entidad_numero#5, denominacion_entidad#6, moneda#7, tipo_de_cambio#8, unidad_de_medida#9, codigo#10, descripcion#11, cantidad#12, valor_unitario#13, precio_unitario#14, descuento#15, subtotal#781, tipo_de_igv#17, igv#18, tipo_de_isc#19, isc#20, impuesto_bolsas#21, total#22

## 7. Paso de DataFrame a RDD — Procesamiento tipo MapReduce

Convertimos la columna de descripción/denominación a un RDD y aplicamos la lógica `flatMap → map → reduceByKey` para contar las palabras más frecuentes en los nombres de productos.

> **¿Cuándo usar RDD vs DataFrame?**
> - **DataFrame**: operaciones estructuradas, SQL, aggregaciones → más eficiente (Catalyst)
> - **RDD**: lógica compleja por fila, procesamiento textual no estructurado, lambda arbitrario

In [58]:
# Convertir la columna de descripción/denominación a RDD
if DESC_COL:
    rdd_textos = df_clean.select(DESC_COL).rdd.map(lambda x: x[0] or "")
    print(f"✅ RDD creado desde columna '{DESC_COL}'")
    print(f"   Primeras 5 entradas:")
    for t in rdd_textos.take(5):
        print(f"   → {t}")
else:
    # Si no hay columna de descripción, usamos otra columna de texto
    col_texto_alt = [c for c in df_clean.columns if df_clean.schema[c].dataType == StringType()]
    if col_texto_alt:
        rdd_textos = df_clean.select(col_texto_alt[0]).rdd.map(lambda x: x[0] or "")
        print(f"✅ RDD creado desde columna '{col_texto_alt[0]}'")
    else:
        print("⚠️  No se encontró columna de texto para el RDD")
        rdd_textos = None

✅ RDD creado desde columna 'descripcion'
   Primeras 5 entradas:
   → PEPSI 355 ML SABOR INTENSO - BOTELLA 
   → CORONA EXTRA BOTELLA 355 ML. SIXPACK 
   → PEPSI 3000 ML - BOTELLA 
   → PEPSI 355 ML SABOR INTENSO - BOTELLA 
   → PEPSI 355 ML SABOR INTENSO - BOTELLA 


26/04/14 13:24:40 WARN TaskSetManager: Stage 84 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


## 8. Filtrar registros por palabra clave (RDD)

Buscamos productos o descripciones que contengan una palabra específica — similar al filtro de "amor" en el dataset bíblico del template original, pero aplicado a nuestras ventas.

In [61]:
if rdd_textos:
    # Filtrar por palabra clave relevante al negocio
    # Puedes cambiar esta palabra por cualquier término de tus productos
    PALABRA_CLAVE = "PEPSI"  # ajusta según NOMBRES DE ENTIDAD

    rdd_filtrado = rdd_textos.filter(
        lambda linea: PALABRA_CLAVE.lower() in str(linea).lower()
    )

    resultados = rdd_filtrado.take(10)
    total_matches = rdd_filtrado.count()

    print(f"🔍 Registros con '{PALABRA_CLAVE}': {total_matches:,}")
    print(f"\nPrimeros 10 ejemplos:")
    for i, t in enumerate(resultados, 1):
        print(f"  {i}. {str(t)[:80]}")

26/04/14 13:24:57 WARN TaskSetManager: Stage 95 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:24:57 WARN TaskSetManager: Stage 96 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


🔍 Registros con 'PEPSI': 276,816

Primeros 10 ejemplos:
  1. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  2. PEPSI 3000 ML - BOTELLA 
  3. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  4. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  5. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  6. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  7. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  8. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  9. PEPSI 355 ML SABOR INTENSO - BOTELLA 
  10. PEPSI 355 ML SABOR INTENSO - BOTELLA 


## 9. flatMap + map + reduceByKey — Word Count sobre Productos

Contamos las **palabras más frecuentes** en los nombres/descripciones de productos. Este es el patrón MapReduce clásico aplicado a datos de ventas reales.

In [62]:
if rdd_textos:
    # Stopwords básicas en español para filtrar ruido
    STOPWORDS = {
        'de', 'la', 'el', 'en', 'y', 'a', 'los', 'las', 'del', 'con',
        'un', 'una', 'por', 'para', 'que', 'se', 'su', 'al', 'es',
        'no', 'lo', 'le', 'o', 'e', 'mas', 'pero', 'si', 'n', 'x'
    }

    # Pipeline MapReduce
    palabras_prod = (
        rdd_textos
        # flatMap: cada línea → lista de palabras limpias
        .flatMap(lambda linea:
            re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", str(linea).lower()).split()
        )
        # filter: quitar stopwords y palabras muy cortas
        .filter(lambda p: p and len(p) > 2 and p not in STOPWORDS)
        # map: crear pares (palabra, 1)
        .map(lambda palabra: (palabra, 1))
        # reduceByKey: sumar conteos por palabra
        .reduceByKey(add)
    )

    top20 = palabras_prod.takeOrdered(20, key=lambda x: -x[1])

    print("🏆 Top 20 palabras más frecuentes en descripciones de productos/ventas:")
    print(f"{'#':>3}  {'Palabra':<25}  {'Frecuencia':>10}")
    print("-" * 45)
    for i, (palabra, freq) in enumerate(top20, 1):
        print(f"{i:>3}. {palabra:<25}  {freq:>10,}")

26/04/14 13:25:03 WARN TaskSetManager: Stage 97 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
[Stage 97:==>                                                     (1 + 27) / 28]

🏆 Top 20 palabras más frecuentes en descripciones de productos/ventas:
  #  Palabra                    Frecuencia
---------------------------------------------
  1. pepsi                         276,823
  2. botella                       236,145
  3. pqt                           225,929
  4. 355                           136,299
  5. 750                            91,770
  6. sabor                          85,670
  7. intenso                        85,661
  8. 2000                           80,678
  9. 500                            66,984
 10. gatorade                       63,249
 11. agua                           59,942
 12. san                            59,044
 13. carlos                         59,006
 14. tropical                       30,365
 15. tapa                           24,371
 16. normal                         24,370
 17. azul                           22,318
 18. concordia                      22,235
 19. 1000                           21,519
 20. paquete           

## 10. Aggregaciones Batch sobre Ventas — Reto Práctico

Ahora combinamos todo lo aprendido para responder preguntas reales de negocio usando el dataset limpio.

**Preguntas a responder:**
1. ¿Cuántas ventas y qué monto total por año?
2. ¿Cuáles son los 3 meses con más ventas históricamente?
3. ¿Cuántas veces aparece una palabra clave en las descripciones? (via RDD)

In [63]:
# ── Pregunta 1: Ventas por año ──────────────────────────────────────────────
if TOTAL_COL and 'anio' in df_clean.columns:
    ventas_por_anio = (
        df_clean
        .groupBy("anio")
        .agg(
            F.count("*").alias("num_facturas"),
            F.sum(TOTAL_COL).alias("total_ventas_soles"),
            F.avg(TOTAL_COL).alias("promedio_factura")
        )
        .orderBy("anio")
        .filter(F.col("anio").isNotNull())
    )

    print("📊 Pregunta 1: Ventas por año")
    print("=" * 55)
    ventas_por_anio.show(truncate=False)

📊 Pregunta 1: Ventas por año


26/04/14 13:25:12 WARN TaskSetManager: Stage 99 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+----+------------+--------------------+------------------+
|anio|num_facturas|total_ventas_soles  |promedio_factura  |
+----+------------+--------------------+------------------+
|2021|126411      |1982272.8651801676  |15.681173831234368|
|2022|122011      |1935084.8155301483  |15.859920954095518|
|2023|98207       |4044018.941124716   |41.17852027986514 |
|2024|98740       |5121554.134994687   |51.86909190798751 |
|2025|72328       |1.5418117009545203E7|213.16940893630687|
|2026|4991        |2195580.653644212   |439.90796506596115|
+----+------------+--------------------+------------------+



In [64]:
# ── Pregunta 2: Top 3 meses con más ventas históricamente ──────────────────
if TOTAL_COL and 'anio_mes' in df_clean.columns:
    top_meses = (
        df_clean
        .groupBy("anio_mes")
        .agg(
            F.sum(TOTAL_COL).alias("total_ventas"),
            F.count("*").alias("num_facturas")
        )
        .filter(F.col("anio_mes").isNotNull())
        .orderBy(F.desc("total_ventas"))
        .limit(3)
    )

    print("🏆 Pregunta 2: Top 3 meses con más ventas")
    print("=" * 45)
    top_meses.show(truncate=False)

🏆 Pregunta 2: Top 3 meses con más ventas


26/04/14 13:25:15 WARN TaskSetManager: Stage 102 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


+--------+------------------+------------+
|anio_mes|total_ventas      |num_facturas|
+--------+------------------+------------+
|2025-11 |4991994.788954409 |4594        |
|2025-10 |3292460.213068129 |2957        |
|2025-12 |2816083.5604204927|4441        |
+--------+------------------+------------+



In [66]:
# ── Pregunta 3: Frecuencia de término en descripciones (RDD) ───────────────
if rdd_textos:
    TERMINO = "pepsi" 

    conteo_termino = (
        palabras_prod
        .filter(lambda x: x[0] == TERMINO)
        .collect()
    )

    if conteo_termino:
        print(f"📌 Pregunta 3: El término '{TERMINO}' aparece {conteo_termino[0][1]:,} veces")
    else:
        print(f"📌 Pregunta 3: El término '{TERMINO}' no aparece en las descripciones")
        print(f"   Intenta cambiar TERMINO por una palabra del top20 de arriba.")

📌 Pregunta 3: El término 'pepsi' aparece 276,823 veces


In [68]:
# 1. Ver qué hay en la columna DESC_COL
#print("🔍 Muestras reales de la columna:")
#muestras = df_clean.select(DESC_COL).limit(10).collect()
#for i, row in enumerate(muestras, 1):
#    print(f"{i}. '{row[0]}'")

# 2. Buscar específicamente "pepsi" en el DataFrame original
#from pyspark.sql.functions import col, lower

#df_con_pepsi = df_clean.filter(lower(col(DESC_COL)).contains("PEPSI"))
#cantidad = df_con_pepsi.count()

#print(f"\n🔎 Registros con 'pepsi' (sin filtrar por RDD): {cantidad}")

#if cantidad > 0:
#    print("Ejemplos:")
#    for row in df_con_pepsi.select(DESC_COL).limit(5).collect():
#        print(f"  → {row[0]}")
#else:
#    print("❌ No hay 'pepsi' en la columna. Posibles causas:")
#    print("   1. La columna tiene otro nombre (ej: 'marca', 'producto')")
#    print("   2. Está escrito como 'Pepsi' (mayúsculas) o 'PEPSI'")
#    print("   3. Tiene espacios: ' Pepsi '")
#    print("   4. Está en otra columna (ej: 'nombre_producto')")

## 11. Cachear el DataFrame limpio para reutilización

Persistimos `df_clean` en memoria Spark para que los próximos notebooks (ETL, HDFS) no tengan que repetir el pipeline de limpieza.

In [69]:
# Cache: Spark almacena las particiones en memoria tras la primera acción
df_ventas = df_clean.cache()
total_registros = df_ventas.count()  # trigger del cache

print("✅ df_ventas cacheado en memoria Spark")
print(f"   Total registros válidos : {total_registros:,}")
print(f"   Total columnas          : {len(df_ventas.columns)}")

if FECHA_COL:
    rango = df_ventas.agg(
        F.min(FECHA_COL).alias("desde"),
        F.max(FECHA_COL).alias("hasta")
    ).collect()[0]
    print(f"   Rango de fechas         : {rango['desde']} → {rango['hasta']}")

if TOTAL_COL:
    totales = df_ventas.agg(
        F.sum(TOTAL_COL).alias("total_ventas"),
        F.avg(TOTAL_COL).alias("promedio_venta"),
        F.max(TOTAL_COL).alias("venta_maxima")
    ).collect()[0]
    print(f"   Total ventas (S/)       : {totales['total_ventas']:,.2f}")
    print(f"   Promedio por factura    : {totales['promedio_venta']:,.2f}")
    print(f"   Factura más alta        : {totales['venta_maxima']:,.2f}")

26/04/14 13:26:10 WARN TaskSetManager: Stage 113 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:26:11 WARN TaskSetManager: Stage 114 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.
26/04/14 13:26:11 WARN TaskSetManager: Stage 117 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


✅ df_ventas cacheado en memoria Spark
   Total registros válidos : 522,688
   Total columnas          : 28
   Rango de fechas         : 2021-01-25 → 2026-04-06
   Total ventas (S/)       : 30,696,628.42
   Promedio por factura    : 58.73
   Factura más alta        : 2,711,864.41


26/04/14 13:26:11 WARN TaskSetManager: Stage 120 contains a task of very large size (2522 KiB). The maximum recommended task size is 1000 KiB.


## 12. Cierre — Reflexión sobre Fundamentos

### ¿Qué aprendimos en esta sesión?

| Concepto | Aplicación en este notebook |
|---|---|
| **SparkSession** | Punto de entrada al cluster, configurada con `local[*]` |
| **DataFrame** | Tabla distribuida con schema; base de todo el análisis batch |
| **Lazy Evaluation** | `.select()`, `.filter()`, `.withColumn()` no ejecutan — solo definen el plan |
| **Acciones** | `.show()`, `.count()`, `.collect()` disparan la ejecución real |
| **DAG** | Grafo de operaciones que Catalyst optimiza antes de ejecutar |
| **explain()** | Visualiza los 4 planes: Parsed → Analyzed → Optimized → Physical |
| **RDD** | Nivel bajo de Spark; útil para procesamiento textual con lambdas |
| **MapReduce** | `flatMap → map → reduceByKey` = patrón clásico de conteo distribuido |
| **Cache** | `.cache()` persiste particiones en RAM para reutilización eficiente |

### Preguntas para reflexión:
1. ¿En qué paso exacto ejecutó Spark por primera vez en este notebook?
2. ¿Qué diferencia observaste en el `explain()` entre un simple `.select()` y una aggregación con `.groupBy()`?
3. ¿Cuándo conviene usar RDD en lugar de DataFrame?

**Próximo Notebook (03):** ETL completo con Spark — transformaciones avanzadas, joins y escritura de resultados.

---
### Resumen del pipeline ejecutado:

| Paso | Acción | Estado |
|---|---|---|
| 1 | SparkSession configurada | ✅ |
| 2 | Excel → pandas → Spark DataFrame | ✅ |
| 3 | Exploración de estructura y columnas clave | ✅ |
| 4 | Transformaciones vs. Acciones demostradas | ✅ |
| 5 | Limpieza: fechas, totales, columnas temporales | ✅ |
| 6 | Plan de ejecución analizado con `explain()` | ✅ |
| 7 | DataFrame → RDD para procesamiento textual | ✅ |
| 8 | Filtrado por palabra clave con RDD | ✅ |
| 9 | Word Count sobre descripciones (MapReduce) | ✅ |
| 10 | Aggregaciones batch: ventas por año, top meses | ✅ |
| 11 | df_ventas cacheado para próximos notebooks | ✅ |